In [0]:
# CELL 1 — Imports
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, current_timestamp, when, lag
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
 
print("Imports OK")

Imports OK


In [0]:
# Season-Aware Threshold Rules
# Different thresholds for each season to account for natural disease patterns
# Structure: {cluster: {season: threshold}}

THRESHOLD_RULES = {
    # ═══════════════════════════════════════════════════════════════════════
    # HIGH-RISK VECTOR-BORNE DISEASES (Dengue, Chikungunya)
    # ═══════════════════════════════════════════════════════════════════════
    "fever_rash_jointpain": {  # Dengue/Chikungunya triad
        "monsoon": 3.5,     # Peak season - higher tolerance
        "summer": 2.8,      # Pre-monsoon uptick expected
        "winter": 1.5,      # Off-season - very sensitive
        "post_monsoon": 2.0,      # Transition period
        "unknown": 2.0      # Fallback
    },
    
    "fever_rash": {  # Dengue early stage
        "monsoon": 3.2,
        "summer": 2.5,
        "winter": 1.5,
        "post_monsoon": 2.0,
        "unknown": 2.0
    },
    
    "fever_jointpain": {  # Chikungunya/viral arthralgia
        "monsoon": 3.5,
        "summer": 3.0,
        "winter": 2.0,
        "post_monsoon": 2.5,
        "unknown": 2.8
    },
    
    # ═══════════════════════════════════════════════════════════════════════
    # CRITICAL WATERBORNE DISEASES
    # ═══════════════════════════════════════════════════════════════════════
    "cholera_like": {  # Cholera (diarrhea+vomiting+dehydration)
        "monsoon": 2.5,     # Monsoon = contaminated water
        "summer": 1.8,      # Peak season - most sensitive
        "winter": 2.2,      # Lower risk
        "post_monsoon": 2.0,
        "unknown": 1.8
    },
    
    "vomiting_diarrhea": {  # GI outbreak indicator
        "monsoon": 3.5,     # Expected during rains
        "summer": 2.5,      # Food spoilage risk
        "winter": 2.8,
        "post_monsoon": 2.5,
        "unknown": 2.5
    },
    
    "diarrhea_only": {  # Most common GI symptom
        "monsoon": 3.5,
        "summer": 2.8,
        "winter": 3.0,
        "post_monsoon": 2.8,
        "unknown": 2.8
    },
    
    "fever_abdominal": {  # Typhoid/enteric fever
        "monsoon": 3.0,
        "summer": 2.5,
        "winter": 2.2,
        "post_monsoon": 2.5,
        "unknown": 2.5
    },
    
    # ═══════════════════════════════════════════════════════════════════════
    # RESPIRATORY ILLNESSES
    # ═══════════════════════════════════════════════════════════════════════
    "cough_cold": {  # Common cold - peak in winter
        "monsoon": 3.5,
        "summer": 3.8,
        "winter": 2.5,      # Peak season - more sensitive
        "post_monsoon": 3.2,
        "unknown": 3.0
    },
    
    "cough_fever": {  # Influenza-like illness
        "monsoon": 3.2,
        "summer": 3.5,
        "winter": 2.2,      # Flu season
        "post_monsoon": 3.0,
        "unknown": 3.0
    },
    
    "cough_only": {  # Isolated cough
        "monsoon": 3.5,
        "summer": 4.0,
        "winter": 2.8,      # More common in cold weather
        "post_monsoon": 3.5,
        "unknown": 3.2
    },
    
    # ═══════════════════════════════════════════════════════════════════════
    # NON-SPECIFIC FEVER
    # ═══════════════════════════════════════════════════════════════════════
    "fever_only": {  # Non-specific fever
        "monsoon": 3.5,     # Many causes in monsoon
        "summer": 3.0,
        "winter": 2.5,
        "post_monsoon": 3.0,
        "unknown": 3.0
    },
    
    # ═══════════════════════════════════════════════════════════════════════
    # CATCH-ALL (52% of data)
    # ═══════════════════════════════════════════════════════════════════════
    "other": {  # Non-specific symptom combinations
        "monsoon": 3.0,
        "summer": 2.8,
        "winter": 2.5,
        "post_monsoon": 2.8,
        "unknown": 2.5
    }
}

# Default thresholds for missing cluster/season combinations
DEFAULT_THRESHOLD = 4.0

print(f"Season-aware threshold rules loaded: {len(THRESHOLD_RULES)} clusters")
print(f"Each cluster has season-specific thresholds (monsoon, summer, winter, post_monsoon)")
print(f"Default threshold for unknown combinations: {DEFAULT_THRESHOLD}")

Season-aware threshold rules loaded: 12 clusters
Each cluster has season-specific thresholds (monsoon, summer, winter, post_monsoon)
Default threshold for unknown combinations: 4.0


In [0]:
# Load Silver Aggregated Table
silver = spark.table("workspace.default.epi_alert_silver_agg").toPandas()

# Check actual columns first
print("Columns in table:", silver.columns.tolist())

# Convert date column safely
silver["update_date"] = pd.to_datetime(silver["update_date"], errors="coerce")

# Drop bad/null dates
silver = silver.dropna(subset=["update_date"])

# Sort rows
silver = silver.sort_values(
    by=["pincode", "symptom_cluster", "update_date"]
).reset_index(drop=True)

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le_season = LabelEncoder()
le_cluster = LabelEncoder()

silver["season"] = silver["season"].fillna("unknown").astype(str)
silver["symptom_cluster"] = silver["symptom_cluster"].fillna("other").astype(str)

silver["season_enc"] = le_season.fit_transform(silver["season"])
silver["cluster_enc"] = le_cluster.fit_transform(silver["symptom_cluster"])

# Features for ML
FEATURE_COLS = [
    "daily_count",
    "7d_avg",
    "30d_avg",
    "count_vs_30d_avg",
    "delta_7d",
    "rate_of_change",
    "season_enc",
    "month",
    "week_of_year",
    "is_weekend",
    "day_of_week"
]

# Fill missing numeric values
for col_name in FEATURE_COLS:
    silver[col_name] = pd.to_numeric(silver[col_name], errors="coerce").fillna(0)

# Final check
print(f"Loaded {len(silver)} rows")
print(f"Clusters: {silver['symptom_cluster'].nunique()}")
print("Unique clusters:")
print(silver["symptom_cluster"].unique())

# Preview data
display(silver.head(10))

Columns in table: ['update_date', 'pincode', 'symptom_cluster', 'season', 'month', 'daily_count', '7d_avg', '30d_avg', 'count_vs_30d_avg', 'prev_day_count', 'delta_7d', 'rate_of_change', 'day_of_week', 'is_weekend', 'week_of_year']
Loaded 57448 rows
Clusters: 12
Unique clusters:
['cholera_like' 'cough_cold' 'cough_fever' 'cough_only' 'diarrhea_only'
 'fever_abdominal' 'fever_jointpain' 'fever_only' 'fever_rash' 'other'
 'vomiting_diarrhea' 'fever_rash_jointpain']


update_date,pincode,symptom_cluster,season,month,daily_count,7d_avg,30d_avg,count_vs_30d_avg,prev_day_count,delta_7d,rate_of_change,day_of_week,is_weekend,week_of_year,season_enc,cluster_enc
2024-05-16T00:00:00.000Z,160001,cholera_like,summer,5,1,1.0,1.0,0.9990009990009991,null,0.0,0.0,5,0,20,2,0
2023-01-05T00:00:00.000Z,160001,cough_cold,winter,1,2,2.0,2.0,0.9995002498750625,null,0.0,0.0,5,0,1,3,1
2023-01-07T00:00:00.000Z,160001,cough_cold,winter,1,1,1.5,1.5,0.6662225183211193,2.0,-0.5,0.49975012493753124,7,1,1,3,1
2023-01-08T00:00:00.000Z,160001,cough_cold,winter,1,2,1.6666666666666667,1.6666666666666667,1.1992804317409556,1.0,0.33333333333333326,1.9980019980019983,1,1,1,3,1
2023-01-09T00:00:00.000Z,160001,cough_cold,winter,1,1,1.5,1.5,0.6662225183211193,2.0,-0.5,0.49975012493753124,2,0,2,3,1
2023-01-12T00:00:00.000Z,160001,cough_cold,winter,1,1,1.4,1.4,0.7137758743754462,1.0,-0.3999999999999999,0.9990009990009991,5,0,2,3,1
2023-01-14T00:00:00.000Z,160001,cough_cold,winter,1,2,1.5,1.5,1.3324450366422387,1.0,0.5,1.9980019980019983,7,1,2,3,1
2023-01-16T00:00:00.000Z,160001,cough_cold,winter,1,1,1.4285714285714286,1.4285714285714286,0.699510342760068,2.0,-0.4285714285714286,0.49975012493753124,2,0,3,3,1
2023-01-19T00:00:00.000Z,160001,cough_cold,winter,1,2,1.4285714285714286,1.5,1.3324450366422387,1.0,0.5714285714285714,1.9980019980019983,5,0,3,3,1
2023-01-20T00:00:00.000Z,160001,cough_cold,winter,1,1,1.4285714285714286,1.4444444444444444,0.6918287339534169,2.0,-0.4285714285714286,0.49975012493753124,6,0,3,3,1


In [0]:
# Validate season-aware threshold rules match actual data
actual_clusters = set(silver["symptom_cluster"].unique())
actual_seasons = set(silver["season"].unique())
rule_clusters = set(THRESHOLD_RULES.keys())

print("\n=== SEASON-AWARE THRESHOLD VALIDATION ===")
print(f"Clusters in data: {len(actual_clusters)}")
print(f"Seasons in data: {len(actual_seasons)} - {sorted(actual_seasons)}")
print(f"Clusters in rules: {len(rule_clusters)}")

# Check for mismatches
missing_rules = actual_clusters - rule_clusters
extra_rules = rule_clusters - actual_clusters

if missing_rules:
    print(f"\n⚠️  WARNING: {len(missing_rules)} cluster(s) in data but NOT in threshold rules:")
    for c in missing_rules:
        print(f"   • {c}")
    print(f"   These will use default threshold = {DEFAULT_THRESHOLD}")

if extra_rules:
    print(f"\nℹ️  INFO: {len(extra_rules)} cluster(s) in rules but NOT in data:")
    for c in extra_rules:
        print(f"   • {c}")
    print("   These rules will not be applied (harmless)")

if not missing_rules and not extra_rules:
    print("\n✅ Perfect match! All clusters have threshold rules.")

# Check season coverage
print("\n=== SEASON COVERAGE CHECK ===")
for cluster in sorted(actual_clusters):
    if cluster in THRESHOLD_RULES:
        cluster_seasons = set(THRESHOLD_RULES[cluster].keys())
        missing_seasons = actual_seasons - cluster_seasons
        if missing_seasons:
            print(f"⚠️  {cluster}: Missing thresholds for seasons {missing_seasons}")
        else:
            print(f"✓ {cluster}: All seasons covered")
    else:
        print(f"❌ {cluster}: No threshold rules defined")

# Show detailed threshold matrix
print("\n=== SEASON-AWARE THRESHOLD MATRIX ===")
print(f"{'Cluster':<25s} | {'Monsoon':>8s} | {'Summer':>8s} | {'Winter':>8s} | {'Post-Mon':>8s} | {'Unknown':>8s} | Rows | Cases | Max Spike")
print("-" * 130)

for cluster in sorted(actual_clusters):
    # Get thresholds
    if cluster in THRESHOLD_RULES:
        monsoon_t = THRESHOLD_RULES[cluster].get("monsoon", DEFAULT_THRESHOLD)
        summer_t = THRESHOLD_RULES[cluster].get("summer", DEFAULT_THRESHOLD)
        winter_t = THRESHOLD_RULES[cluster].get("winter", DEFAULT_THRESHOLD)
        post_monsoon_t = THRESHOLD_RULES[cluster].get("post_monsoon", DEFAULT_THRESHOLD)
        unknown_t = THRESHOLD_RULES[cluster].get("unknown", DEFAULT_THRESHOLD)
    else:
        monsoon_t = summer_t = winter_t = post_monsoon_t = unknown_t = DEFAULT_THRESHOLD
    
    # Get data stats
    cluster_data = silver[silver["symptom_cluster"] == cluster]
    count = len(cluster_data)
    cases = cluster_data["daily_count"].sum()
    max_spike = cluster_data["count_vs_30d_avg"].max()
    
    print(f"{cluster:<25s} | {monsoon_t:>8.1f} | {summer_t:>8.1f} | {winter_t:>8.1f} | {post_monsoon_t:>8.1f} | {unknown_t:>8.1f} | {count:>4d} | {cases:>5.0f} | {max_spike:>9.2f}x")

print("\nℹ️  Lower thresholds = more sensitive (flags smaller spikes)")
print("ℹ️  Higher thresholds = less sensitive (only flags large spikes)")
print(f"ℹ️  Default threshold for missing combinations: {DEFAULT_THRESHOLD}")


=== SEASON-AWARE THRESHOLD VALIDATION ===
Clusters in data: 12
Seasons in data: 4 - ['monsoon', 'post_monsoon', 'summer', 'winter']
Clusters in rules: 12

✅ Perfect match! All clusters have threshold rules.

=== SEASON COVERAGE CHECK ===
✓ cholera_like: All seasons covered
✓ cough_cold: All seasons covered
✓ cough_fever: All seasons covered
✓ cough_only: All seasons covered
✓ diarrhea_only: All seasons covered
✓ fever_abdominal: All seasons covered
✓ fever_jointpain: All seasons covered
✓ fever_only: All seasons covered
✓ fever_rash: All seasons covered
✓ fever_rash_jointpain: All seasons covered
✓ other: All seasons covered
✓ vomiting_diarrhea: All seasons covered

=== SEASON-AWARE THRESHOLD MATRIX ===
Cluster                   |  Monsoon |   Summer |   Winter | Post-Mon |  Unknown | Rows | Cases | Max Spike
----------------------------------------------------------------------------------------------------------------------------------
cholera_like              |      2.5 |      1.8

In [0]:
# CELL 6 — Train Isolation Forest model with MLflow tracking
import json
import tempfile
import os
import mlflow.models

# Set MLflow experiment (use workspace path that auto-creates)
mlflow.set_experiment("/Users/cse230001056@iiti.ac.in/EpiAlert_Anomaly_Detection")

# Start MLflow run
with mlflow.start_run(run_name="isolation_forest_with_season_thresholds") as run:
    
    # ═══════════════════════════════════════════════════════════════
    # 1. LOG PARAMETERS
    # ═══════════════════════════════════════════════════════════════
    contamination_rate = 0.05
    n_estimators_val = 100
    random_state_val = 42
    
    mlflow.log_param("model_type", "IsolationForest")
    mlflow.log_param("contamination", contamination_rate)
    mlflow.log_param("n_estimators", n_estimators_val)
    mlflow.log_param("random_state", random_state_val)
    mlflow.log_param("max_samples", "auto")
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("feature_cols", ",".join(FEATURE_COLS))
    mlflow.log_param("n_symptom_clusters", silver["symptom_cluster"].nunique())
    mlflow.log_param("training_rows", len(silver))
    mlflow.log_param("threshold_strategy", "season_aware")
    mlflow.log_param("default_threshold", DEFAULT_THRESHOLD)
    
    # Log season-aware threshold rules (flattened for MLflow)
    for cluster, season_thresholds in THRESHOLD_RULES.items():
        for season, threshold in season_thresholds.items():
            mlflow.log_param(f"threshold_{cluster}_{season}", threshold)
    
    # ═══════════════════════════════════════════════════════════════
    # 2. TRAIN ISOLATION FOREST
    # ═══════════════════════════════════════════════════════════════
    X = silver[FEATURE_COLS].values
    
    print("Training Isolation Forest model...")
    if_model = IsolationForest(
        contamination=contamination_rate,
        random_state=random_state_val,
        n_estimators=n_estimators_val,
        max_samples='auto'
    )
    if_model.fit(X)
    
    # Get predictions and scores
    predictions = if_model.predict(X)  # -1 for anomaly, 1 for normal
    scores = if_model.score_samples(X)  # lower = more anomalous
    silver["if_anomaly"] = (predictions == -1).astype(int)  # convert to 0/1
    silver["if_score"] = scores
    
    if_count = silver['if_anomaly'].sum()
    if_pct = silver['if_anomaly'].mean() * 100
    print(f"IF detected {if_count} anomalies ({if_pct:.2f}%)")
    
    # ═══════════════════════════════════════════════════════════════
    # 3. APPLY SEASON-AWARE THRESHOLD RULES
    # ═══════════════════════════════════════════════════════════════
    def apply_threshold(row):
        """Apply season-aware threshold based on cluster and season."""
        cluster = row["symptom_cluster"]
        season = row["season"]
        
        # Lookup threshold with fallbacks
        if cluster in THRESHOLD_RULES:
            if season in THRESHOLD_RULES[cluster]:
                threshold = THRESHOLD_RULES[cluster][season]
            else:
                # Season not found, use 'unknown' fallback or default
                threshold = THRESHOLD_RULES[cluster].get("unknown", DEFAULT_THRESHOLD)
        else:
            # Cluster not found, use default
            threshold = DEFAULT_THRESHOLD
        
        return 1 if row["count_vs_30d_avg"] > threshold else 0
    
    silver["threshold_flag"] = silver.apply(apply_threshold, axis=1)
    threshold_count = silver['threshold_flag'].sum()
    threshold_pct = silver['threshold_flag'].mean() * 100
    print(f"Season-aware threshold flagged {threshold_count} anomalies ({threshold_pct:.2f}%)")
    
    # Combined flag: either IF or threshold triggers
    silver["combined_flag"] = ((silver["if_anomaly"] == 1) | (silver["threshold_flag"] == 1)).astype(int)
    combined_count = silver['combined_flag'].sum()
    combined_pct = silver['combined_flag'].mean() * 100
    print(f"Combined: {combined_count} anomalies ({combined_pct:.2f}%)")
    
    # ═══════════════════════════════════════════════════════════════
    # 4. LOG METRICS
    # ═══════════════════════════════════════════════════════════════
    mlflow.log_metric("if_anomaly_count", if_count)
    mlflow.log_metric("if_anomaly_pct", if_pct)
    mlflow.log_metric("threshold_anomaly_count", threshold_count)
    mlflow.log_metric("threshold_anomaly_pct", threshold_pct)
    mlflow.log_metric("combined_anomaly_count", combined_count)
    mlflow.log_metric("combined_anomaly_pct", combined_pct)
    mlflow.log_metric("mean_anomaly_score", silver["if_score"].mean())
    mlflow.log_metric("min_anomaly_score", silver["if_score"].min())
    mlflow.log_metric("max_anomaly_score", silver["if_score"].max())
    
    # Cluster-level metrics
    for cluster in silver["symptom_cluster"].unique():
        cluster_data = silver[silver["symptom_cluster"] == cluster]
        cluster_anomalies = cluster_data["combined_flag"].sum()
        mlflow.log_metric(f"anomalies_{cluster}", cluster_anomalies)
    
    # Season-level metrics
    for season in silver["season"].unique():
        season_data = silver[silver["season"] == season]
        season_anomalies = season_data["combined_flag"].sum()
        mlflow.log_metric(f"anomalies_season_{season}", season_anomalies)
    
    # ═══════════════════════════════════════════════════════════════
    # 5. LOG MODEL WITH SIGNATURE
    # ═══════════════════════════════════════════════════════════════
    # Create model signature for Unity Catalog
    signature = mlflow.models.infer_signature(
        X[:100],  # Sample of input features
        predictions[:100]  # Sample of predictions
    )
    
    mlflow.sklearn.log_model(
        if_model,
        artifact_path="isolation_forest_model",
        signature=signature,
        registered_model_name="EpiAlert_IsolationForest"
    )
    
    # ═══════════════════════════════════════════════════════════════
    # 6. LOG ARTIFACTS
    # ═══════════════════════════════════════════════════════════════
    # Save season-aware threshold rules as JSON
    with tempfile.TemporaryDirectory() as tmpdir:
        threshold_file = os.path.join(tmpdir, "threshold_rules_season_aware.json")
        with open(threshold_file, "w") as f:
            json.dump(THRESHOLD_RULES, f, indent=2)
        mlflow.log_artifact(threshold_file)
        
        # Save feature names
        feature_file = os.path.join(tmpdir, "features.json")
        with open(feature_file, "w") as f:
            json.dump({"features": FEATURE_COLS}, f, indent=2)
        mlflow.log_artifact(feature_file)
    
    # Log run info
    print(f"\n✓ MLflow Run ID: {run.info.run_id}")
    print(f"✓ Experiment ID: {run.info.experiment_id}")
    print(f"✓ Model registered as: EpiAlert_IsolationForest")
    print(f"✓ Using season-aware thresholds")

# Rename update_date to report_date for consistency with downstream code
anomaly_df = silver.copy()
anomaly_df = anomaly_df.rename(columns={"update_date": "report_date"})

print(f"\nCreated anomaly_df with {len(anomaly_df)} rows")
print(f"Columns: {anomaly_df.columns.tolist()}")

Training Isolation Forest model...
IF detected 2873 anomalies (5.00%)
Season-aware threshold flagged 55 anomalies (0.10%)
Combined: 2873 anomalies (5.00%)


2026/04/18 06:05:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0e6510d5-28f5.cloud.databricks.com/ml/experiments/4108542647799536/models/m-dfe1dadd969d43d39d2bfe694d6077cb?o=7474657722255430
Registered model 'EpiAlert_IsolationForest' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.epialert_isolationforest': https://dbc-0e6510d5-28f5.cloud.databricks.com/explore/data/models/workspace/default/epialert_isolationforest/version/3?o=7474657722255430



✓ MLflow Run ID: 176fe3ddd7494f22a5a33c8b773922e8
✓ Experiment ID: 4108542647799536
✓ Model registered as: EpiAlert_IsolationForest
✓ Using season-aware thresholds

Created anomaly_df with 57448 rows
Columns: ['report_date', 'pincode', 'symptom_cluster', 'season', 'month', 'daily_count', '7d_avg', '30d_avg', 'count_vs_30d_avg', 'prev_day_count', 'delta_7d', 'rate_of_change', 'day_of_week', 'is_weekend', 'week_of_year', 'season_enc', 'cluster_enc', 'if_anomaly', 'if_score', 'threshold_flag', 'combined_flag']


In [0]:
# Display MLflow Experiment Info
import mlflow

experiment = mlflow.get_experiment_by_name("/Users/cse230001056@iiti.ac.in/EpiAlert_Anomaly_Detection")

if experiment:
    print("="*70)
    print("MLflow Experiment Tracking - EpiAlert Anomaly Detection")
    print("="*70)
    print(f"Experiment Name: {experiment.name}")
    print(f"Experiment ID: {experiment.experiment_id}")
    print(f"Artifact Location: {experiment.artifact_location}")
    print(f"\nAccess MLflow UI: Databricks Workspace → Experiments → EpiAlert_Anomaly_Detection")
    print("\nLogged Information:")
    print("  • Model: Isolation Forest (registered in Unity Catalog)")
    print("  • Parameters: contamination, n_estimators, threshold rules")
    print("  • Metrics: anomaly counts, percentages, per-cluster stats")
    print("  • Artifacts: threshold_rules.json, features.json")
    print("="*70)
    
    # Get latest run info
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=1
    )
    
    if not runs.empty:
        latest_run = runs.iloc[0]
        print(f"\nLatest Run:")
        print(f"  Run ID: {latest_run['run_id']}")
        print(f"  Status: {latest_run['status']}")
        print(f"  IF Anomalies: {latest_run['metrics.if_anomaly_count']:.0f} ({latest_run['metrics.if_anomaly_pct']:.2f}%)")
        print(f"  Threshold Anomalies: {latest_run['metrics.threshold_anomaly_count']:.0f} ({latest_run['metrics.threshold_anomaly_pct']:.2f}%)")
        print(f"  Combined Anomalies: {latest_run['metrics.combined_anomaly_count']:.0f} ({latest_run['metrics.combined_anomaly_pct']:.2f}%)")
else:
    print("Experiment not found. Run the previous cell to create it.")

MLflow Experiment Tracking - EpiAlert Anomaly Detection
Experiment Name: /Users/cse230001056@iiti.ac.in/EpiAlert_Anomaly_Detection
Experiment ID: 4108542647799536
Artifact Location: dbfs:/databricks/mlflow-tracking/4108542647799536

Access MLflow UI: Databricks Workspace → Experiments → EpiAlert_Anomaly_Detection

Logged Information:
  • Model: Isolation Forest (registered in Unity Catalog)
  • Parameters: contamination, n_estimators, threshold rules
  • Metrics: anomaly counts, percentages, per-cluster stats
  • Artifacts: threshold_rules.json, features.json

Latest Run:
  Run ID: 176fe3ddd7494f22a5a33c8b773922e8
  Status: FINISHED
  IF Anomalies: 2873 (5.00%)
  Threshold Anomalies: 55 (0.10%)
  Combined Anomalies: 2873 (5.00%)


In [0]:
# COMMAND ----------
# CELL 5 — Persistence filter: require 2+ consecutive anomaly days
# This removes isolated one-day spikes (noise)
 
anomaly_df = anomaly_df.sort_values(["pincode","symptom_cluster","report_date"])
anomaly_df["prev_flag"] = anomaly_df.groupby(
    ["pincode","symptom_cluster"])["combined_flag"].shift(1).fillna(0)
anomaly_df["persistent_anomaly"] = (
    (anomaly_df["combined_flag"] == 1) & (anomaly_df["prev_flag"] == 1)
).astype(int)
 
# Rising trend: delta_7d > 0 on same day
anomaly_df["rising_trend"] = (anomaly_df["delta_7d"] > 0).astype(int)
 
# Final alert flag: persistent anomaly OR (combined_flag AND rising_trend)
anomaly_df["alert_flag"] = (
    (anomaly_df["persistent_anomaly"] == 1) |
    ((anomaly_df["combined_flag"] == 1) & (anomaly_df["rising_trend"] == 1))
).astype(int)
 
# Risk level label
def risk_level(row):
    score = -row["if_score"]   # invert: higher = more anomalous
    if row["alert_flag"] == 0:
        return "normal"
    elif score > 0.15 or row["count_vs_30d_avg"] > 4:
        return "high"
    elif score > 0.05 or row["count_vs_30d_avg"] > 2.5:
        return "medium"
    else:
        return "low"
 
anomaly_df["risk_level"] = anomaly_df.apply(risk_level, axis=1)
 
print("Alert flag distribution:")
print(anomaly_df["alert_flag"].value_counts())
print("\nRisk level distribution:")
print(anomaly_df["risk_level"].value_counts())
 
# COMMAND ----------
# CELL 6 — Write Gold anomaly table
gold_schema_df = spark.createDataFrame(anomaly_df[[
    "report_date","pincode","symptom_cluster","season","month",
    "week_of_year","day_of_week","is_weekend",
    "daily_count","7d_avg","30d_avg","count_vs_30d_avg",
    "prev_day_count","delta_7d","rate_of_change",
    "if_score","if_anomaly","threshold_flag",
    "combined_flag","persistent_anomaly","rising_trend",
    "alert_flag","risk_level"
]])
 
spark.sql("""
CREATE TABLE IF NOT EXISTS epi_alert_gold_anomaly (
  report_date DATE,
  pincode INT,
  symptom_cluster STRING,
  season STRING,
  month INT,
  week_of_year INT,
  day_of_week INT,
  is_weekend INT,
  daily_count DOUBLE,
  `7d_avg` DOUBLE,
  `30d_avg` DOUBLE,
  count_vs_30d_avg DOUBLE,
  prev_day_count DOUBLE,
  delta_7d DOUBLE,
  rate_of_change DOUBLE,
  if_score DOUBLE,
  if_anomaly INT,
  threshold_flag INT,
  combined_flag INT,
  persistent_anomaly INT,
  rising_trend INT,
  alert_flag INT,
  risk_level STRING
)
USING DELTA
""")
 
gold_schema_df.write.format("delta").mode("overwrite").saveAsTable("epi_alert_gold_anomaly")
 
print(f"✓ Gold table written: epi_alert_gold_anomaly")
print(f"✓ Total rows: {len(anomaly_df)}")
print(f"✓ Alert rows: {anomaly_df['alert_flag'].sum()}")
print(f"✓ Risk breakdown: {anomaly_df['risk_level'].value_counts().to_dict()}")

Alert flag distribution:
alert_flag
0    54800
1     2648
Name: count, dtype: int64

Risk level distribution:
risk_level
normal    54800
high       2648
Name: count, dtype: int64
✓ Gold table written: epi_alert_gold_anomaly
✓ Total rows: 57448
✓ Alert rows: 2648
✓ Risk breakdown: {'normal': 54800, 'high': 2648}


In [0]:
# CELL 7 — Preview active alerts
spark.sql("""
    SELECT report_date, pincode, symptom_cluster,
           daily_count, ROUND(`30d_avg`,1) AS baseline,
           ROUND(count_vs_30d_avg,2) AS ratio,
           risk_level
    FROM   epi_alert_gold_anomaly
    WHERE  alert_flag = 1
    ORDER  BY report_date DESC, count_vs_30d_avg DESC
    LIMIT  20
""").show(truncate=False)

+-------------------+-------+---------------+-----------+--------+-----+----------+
|report_date        |pincode|symptom_cluster|daily_count|baseline|ratio|risk_level|
+-------------------+-------+---------------+-----------+--------+-----+----------+
|2024-12-31 00:00:00|522020 |cough_only     |3          |1.4     |2.19 |high      |
|2024-12-31 00:00:00|797003 |cough_only     |3          |1.4     |2.14 |high      |
|2024-12-31 00:00:00|700001 |other          |3          |1.9     |1.61 |high      |
|2024-12-31 00:00:00|171001 |other          |3          |2.1     |1.45 |high      |
|2024-12-31 00:00:00|171002 |other          |2          |2.1     |0.94 |high      |
|2024-12-30 00:00:00|302003 |other          |3          |1.3     |2.31 |high      |
|2024-12-30 00:00:00|560002 |cough_only     |3          |1.4     |2.19 |high      |
|2024-12-30 00:00:00|171002 |other          |3          |2.1     |1.41 |high      |
|2024-12-29 00:00:00|171001 |cough_only     |3          |1.3     |2.37 |high

In [0]:
# ═══════════════════════════════════════════════════════════════════════════
# DISEASE PREDICTION MODEL - Random Forest Multi-Label Classifier
# Predicts disease probabilities directly from symptom combinations
# ═══════════════════════════════════════════════════════════════════════════

import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss
import json

# ───────────────────────────────────────────────────────────────────────────
# 1. DISEASE-TO-SYMPTOM MAPPING (from forecasting notebook)
# ───────────────────────────────────────────────────────────────────────────

# Comprehensive disease definitions based on symptom patterns
DISEASE_SYMPTOM_PROFILES = {
    "dengue": ["fever", "rash", "joint_pain", "headache", "body_ache"],
    "chikungunya": ["fever", "joint_pain", "rash"],
    "malaria": ["fever", "headache", "body_ache", "chills"],
    "cholera": ["diarrhea", "vomiting", "dehydration"],
    "typhoid": ["fever", "abdominal_pain", "diarrhea", "headache"],
    "gastroenteritis": ["diarrhea", "vomiting", "abdominal_pain"],
    "influenza": ["fever", "cough", "body_ache", "headache"],
    "common_cold": ["cough", "cold", "runny_nose"],
    "pneumonia": ["cough", "fever", "breathing_difficulty"],
    "covid_like": ["fever", "cough", "breathing_difficulty"],
    "measles": ["fever", "rash", "cough"],
    "leptospirosis": ["fever", "headache", "body_ache", "conjunctival_redness"],
    "food_poisoning": ["vomiting", "diarrhea", "abdominal_pain"],
}

# All possible symptoms in the dataset
ALL_SYMPTOMS = [
    "fever", "cough", "cold", "rash", "joint_pain", "diarrhea", 
    "vomiting", "abdominal_pain", "headache", "body_ache", "chills",
    "runny_nose", "breathing_difficulty", "conjunctival_redness", "dehydration"
]

ALL_DISEASES = list(DISEASE_SYMPTOM_PROFILES.keys())

print(f"Diseases to predict: {len(ALL_DISEASES)}")
print(f"Symptom features: {len(ALL_SYMPTOMS)}")

# ───────────────────────────────────────────────────────────────────────────
# 2. CREATE SYNTHETIC TRAINING DATA
# ───────────────────────────────────────────────────────────────────────────

def create_training_data():
    """
    Generate training examples from disease-symptom profiles.
    Each disease generates multiple examples with symptom variations.
    """
    training_data = []
    
    for disease, core_symptoms in DISEASE_SYMPTOM_PROFILES.items():
        # Create 50 examples per disease with variations
        for _ in range(50):
            # Start with core symptoms (90% probability each)
            symptoms_present = [s for s in core_symptoms if np.random.random() < 0.9]
            
            # Add noise: random symptoms with 5% probability
            for symptom in ALL_SYMPTOMS:
                if symptom not in symptoms_present and np.random.random() < 0.05:
                    symptoms_present.append(symptom)
            
            # Create binary feature vector
            features = {symptom: 1 if symptom in symptoms_present else 0 
                       for symptom in ALL_SYMPTOMS}
            
            # Create disease label vector (multi-label: can have multiple diseases)
            labels = {d: 1 if d == disease else 0 for d in ALL_DISEASES}
            
            training_data.append({**features, **labels, 'primary_disease': disease})
    
    return pd.DataFrame(training_data)

train_df = create_training_data()
print(f"\nTraining data created: {len(train_df)} examples")
print(f"Disease distribution:\n{train_df['primary_disease'].value_counts()}")

# ───────────────────────────────────────────────────────────────────────────
# 3. PREPARE FEATURES AND LABELS
# ───────────────────────────────────────────────────────────────────────────

X = train_df[ALL_SYMPTOMS].values
y = train_df[ALL_DISEASES].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# ───────────────────────────────────────────────────────────────────────────
# 4. TRAIN RANDOM FOREST CLASSIFIER WITH MLFLOW
# ───────────────────────────────────────────────────────────────────────────

mlflow.set_experiment("/Users/cse230001056@iiti.ac.in/EpiAlert_Disease_Prediction")

with mlflow.start_run(run_name="random_forest_disease_classifier") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("n_diseases", len(ALL_DISEASES))
    mlflow.log_param("n_symptoms", len(ALL_SYMPTOMS))
    mlflow.log_param("training_samples", len(X_train))
    mlflow.log_param("diseases", ",".join(ALL_DISEASES))
    mlflow.log_param("symptoms", ",".join(ALL_SYMPTOMS))
    
    # Train model
    print("\n" + "="*70)
    print("Training Random Forest Disease Classifier...")
    print("="*70)
    
    rf_model = MultiOutputClassifier(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
    )
    
    rf_model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = rf_model.predict(X_test)
    
    # Calculate metrics
    hamming = hamming_loss(y_test, y_pred)
    accuracy = (y_pred == y_test).all(axis=1).mean()  # Exact match accuracy
    
    mlflow.log_metric("hamming_loss", hamming)
    mlflow.log_metric("exact_match_accuracy", accuracy)
    
    print(f"\n✓ Model trained successfully")
    print(f"  Hamming Loss: {hamming:.4f}")
    print(f"  Exact Match Accuracy: {accuracy:.4f}")
    
    # Log model with signature
    signature = mlflow.models.infer_signature(X_train[:100], y_pred[:100])
    mlflow.sklearn.log_model(
        rf_model,
        artifact_path="disease_classifier_model",
        signature=signature,
        registered_model_name="EpiAlert_DiseaseClassifier"
    )
    
    # Log disease and symptom mappings as artifacts
    import tempfile
    import os
    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Disease profiles
        disease_file = os.path.join(tmpdir, "disease_profiles.json")
        with open(disease_file, "w") as f:
            json.dump(DISEASE_SYMPTOM_PROFILES, f, indent=2)
        mlflow.log_artifact(disease_file)
        
        # Symptom list
        symptom_file = os.path.join(tmpdir, "symptoms.json")
        with open(symptom_file, "w") as f:
            json.dump({"symptoms": ALL_SYMPTOMS, "diseases": ALL_DISEASES}, f, indent=2)
        mlflow.log_artifact(symptom_file)
    
    print(f"\n✓ MLflow Run ID: {run.info.run_id}")
    print(f"✓ Model registered as: EpiAlert_DiseaseClassifier")

print("\n" + "="*70)
print("Disease Prediction Model Ready ✓")
print("="*70)

Diseases to predict: 13
Symptom features: 15

Training data created: 650 examples
Disease distribution:
primary_disease
dengue             50
chikungunya        50
malaria            50
cholera            50
typhoid            50
gastroenteritis    50
influenza          50
common_cold        50
pneumonia          50
covid_like         50
measles            50
leptospirosis      50
food_poisoning     50
Name: count, dtype: int64

Training set: 520 samples
Test set: 130 samples

Training Random Forest Disease Classifier...


2026/04/18 06:17:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



✓ Model trained successfully
  Hamming Loss: 0.0396
  Exact Match Accuracy: 0.6000


🔗 View Logged Model at: https://dbc-0e6510d5-28f5.cloud.databricks.com/ml/experiments/346393160574590/models/m-0e698c3a184247ce87787fe58c654b6d?o=7474657722255430
Registered model 'EpiAlert_DiseaseClassifier' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.epialert_diseaseclassifier': https://dbc-0e6510d5-28f5.cloud.databricks.com/explore/data/models/workspace/default/epialert_diseaseclassifier/version/2?o=7474657722255430



✓ MLflow Run ID: 879cda1b3c49428c89c87e94d083cc13
✓ Model registered as: EpiAlert_DiseaseClassifier

Disease Prediction Model Ready ✓


In [0]:
# ═══════════════════════════════════════════════════════════════════════════
# APPLY DISEASE PREDICTIONS TO ANOMALY DATA
# Adds disease_prediction column with top 3 diseases and probabilities
# ═══════════════════════════════════════════════════════════════════════════

def extract_symptom_features(symptom_cluster):
    """
    Extract binary symptom features from symptom_cluster string.
    Maps cluster names to individual symptom presence.
    """
    features = {symptom: 0 for symptom in ALL_SYMPTOMS}
    
    cluster_lower = str(symptom_cluster).lower()
    
    # Map cluster names to symptom features
    if "fever" in cluster_lower:
        features["fever"] = 1
    if "rash" in cluster_lower:
        features["rash"] = 1
    if "joint" in cluster_lower or "jointpain" in cluster_lower:
        features["joint_pain"] = 1
    if "cough" in cluster_lower:
        features["cough"] = 1
    if "cold" in cluster_lower:
        features["cold"] = 1
    if "diarrhea" in cluster_lower:
        features["diarrhea"] = 1
    if "vomit" in cluster_lower:
        features["vomiting"] = 1
    if "abdominal" in cluster_lower:
        features["abdominal_pain"] = 1
    if "head" in cluster_lower:
        features["headache"] = 1
    if "body" in cluster_lower:
        features["body_ache"] = 1
    
    # Special patterns
    if "cholera" in cluster_lower:
        features["diarrhea"] = 1
        features["vomiting"] = 1
        features["dehydration"] = 1
    
    return [features[symptom] for symptom in ALL_SYMPTOMS]

# ───────────────────────────────────────────────────────────────────────────
# 1. APPLY TO ANOMALY DATAFRAME
# ───────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("Applying Disease Predictions to Anomaly Data")
print("="*70)

# Extract symptom features from anomaly_df
print("\nExtracting symptom features from symptom_cluster...")
symptom_features = np.array([
    extract_symptom_features(cluster) 
    for cluster in anomaly_df["symptom_cluster"]
])

print(f"Feature matrix shape: {symptom_features.shape}")

# ───────────────────────────────────────────────────────────────────────────
# 2. PREDICT DISEASE PROBABILITIES
# ───────────────────────────────────────────────────────────────────────────

print("\nPredicting disease probabilities...")

# Get probability predictions for each disease
disease_probs = []
for estimator_idx, estimator in enumerate(rf_model.estimators_):
    probs = estimator.predict_proba(symptom_features)
    # Extract probability of class 1 (disease present)
    disease_probs.append(probs[:, 1] if probs.shape[1] > 1 else np.zeros(len(symptom_features)))

disease_prob_matrix = np.array(disease_probs).T  # Shape: (n_samples, n_diseases)

print(f"Disease probability matrix shape: {diseasie_prob_matrix.shape}")

# ───────────────────────────────────────────────────────────────────────────
# 3. FORMAT TOP 3 DISEASES PER ROW
# ───────────────────────────────────────────────────────────────────────────

def format_top_diseases(probs, top_k=3):
    """
    Format top K diseases with probabilities as JSON string.
    """
    top_indices = np.argsort(probs)[::-1][:top_k]
    top_diseases = [
        {
            "disease": ALL_DISEASES[idx],
            "probability": round(float(probs[idx]), 3)
        }
        for idx in top_indices
        if probs[idx] > 0.01  # Only include if >1% probability
    ]
    return json.dumps(top_diseases) if top_diseases else json.dumps([{"disease": "unknown", "probability": 0.0}])

print("\nFormatting disease predictions...")
anomaly_df["disease_prediction"] = [
    format_top_diseases(probs) for probs in disease_prob_matrix
]

# Extract primary disease and probability
anomaly_df["primary_disease"] = [
    json.loads(pred)[0]["disease"] for pred in anomaly_df["disease_prediction"]
]
anomaly_df["primary_disease_prob"] = [
    json.loads(pred)[0]["probability"] for pred in anomaly_df["disease_prediction"]
]

print("\n✓ Disease predictions added to anomaly_df")

# ───────────────────────────────────────────────────────────────────────────
# 4. SUMMARY STATISTICS
# ───────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("Disease Prediction Summary")
print("="*70)

print(f"\nTotal rows predicted: {len(anomaly_df)}")
print(f"\nPrimary Disease Distribution:")
print(anomaly_df["primary_disease"].value_counts().head(10))

print(f"\nAverage prediction confidence: {anomaly_df['primary_disease_prob'].mean():.3f}")

# Show disease distribution for high-risk alerts
high_risk = anomaly_df[anomaly_df["alert_flag"] == 1]
print(f"\n\nDisease Distribution in HIGH-RISK ALERTS ({len(high_risk)} alerts):")
print(high_risk["primary_disease"].value_counts().head(10))

# Preview sample predictions
print("\n\nSample Predictions:")
sample_cols = ["pincode", "symptom_cluster", "primary_disease", "primary_disease_prob", 
               "alert_flag", "risk_level", "daily_count"]
display(anomaly_df[sample_cols].head(20))

print("\n" + "="*70)
print("✓ Disease Prediction Complete")
print("="*70)


Applying Disease Predictions to Anomaly Data

Extracting symptom features from symptom_cluster...
Feature matrix shape: (57448, 15)

Predicting disease probabilities...
Disease probability matrix shape: (57448, 13)

Formatting disease predictions...

✓ Disease predictions added to anomaly_df

Disease Prediction Summary

Total rows predicted: 57448

Primary Disease Distribution:
primary_disease
gastroenteritis    29881
chikungunya        10132
cholera             7816
pneumonia           6185
common_cold         3238
food_poisoning       133
typhoid               63
Name: count, dtype: int64

Average prediction confidence: 0.350


Disease Distribution in HIGH-RISK ALERTS (2648 alerts):
primary_disease
gastroenteritis    2067
pneumonia           212
cholera             174
chikungunya         129
common_cold          66
Name: count, dtype: int64


Sample Predictions:


pincode,symptom_cluster,primary_disease,primary_disease_prob,alert_flag,risk_level,daily_count
160001,cholera_like,cholera,0.997,0,normal,1
160001,cough_cold,common_cold,0.669,0,normal,2
160001,cough_cold,common_cold,0.669,0,normal,1
160001,cough_cold,common_cold,0.669,0,normal,2
160001,cough_cold,common_cold,0.669,0,normal,1
160001,cough_cold,common_cold,0.669,0,normal,1
160001,cough_cold,common_cold,0.669,0,normal,2
160001,cough_cold,common_cold,0.669,0,normal,1
160001,cough_cold,common_cold,0.669,0,normal,2
160001,cough_cold,common_cold,0.669,0,normal,1



✓ Disease Prediction Complete
